# TP 3 - Agent MCP

Ce notebook migre les outils de l'agent vers une architecture **MCP** (Model Context Protocol).
Les outils Python codés en `3_1` sont remplacés par des serveurs MCP distants et un serveur local.
L'agent délègue l'exécution à ces serveurs via PydanticAI, sans accéder directement aux API.

### 0.1. Objectif
- **TP 3_1** : Coder les outils et l'infrastructure d'exécution, puis construire un agent PydanticAI qui les orchestre
- **TP 3_2** : Migrer les outils vers une architecture MCP (serveurs locaux et distants). Seul `rag_search` est à coder dans le serveur local

### 0.2. Documentation générale

[PydanticAI - MCP](https://ai.pydantic.dev/mcp/)

[Model Context Protocol (MCP)](https://modelcontextprotocol.io/introduction)

[DeepWiki MCP](https://deepwiki.com/)

[Tavily MCP](https://docs.tavily.com/docs/tavily-api/mcp)

[OpenMCP - Google Maps](https://mcp.open-mcp.org/)

In [ ]:
import sys
from urllib.parse import urlencode

from pydantic_ai.mcp import MCPServerStdio, MCPServerStreamableHTTP
from pydantic_ai.models.google import GoogleModel, GoogleModelSettings
from pydantic_ai.providers.google import GoogleProvider

from shared.agent_utils import LoggingAgent
from shared.config import ROOT_DIR, project_settings

LOG_DIR = ROOT_DIR / "TP3_travel_planner_Agent" / "logs"

### 0.3. Récapitulatif des fonctions utilisées dans ce notebook

**Fournies**

- `EventStreamHandler` : classe qui formate les appels outils et leurs résultats en temps réel
- `LoggingAgent` : agent PydanticAI avec `run_with_logging` (codé en TP 3_1) pour l'exécution tracée + log JSON

--> Disponibles dans `shared/agent_utils.py`

- `Place` : format de lieu normalisé (nom + coordonnées GPS)
- `tool_get_current_date` : retourne la date courante pour ancrer les dates relatives
- `tool_geocode_location` : convertit un lieu texte en coordonnées GPS
- `tool_get_weather` : retourne une prévision météo sur une plage de dates
- `tool_search_nearby` : cherche des lieux proches autour de coordonnées
- `tool_retrieve_docs` : récupère les passages RAG internes les plus pertinents
- `web_search` : lance une recherche web externe (Tavily Search API)
- `web_extract` : lit le contenu de pages web trouvées (Tavily Extract API)

--> Disponibles dans `shared/agent_tools.py`

**À coder dans ce notebook**

- `rag_search` : recherche vectorielle RAG exposée comme outil MCP local

--> À implémenter dans `TP3_travel_planner_Agent/mcp_server.py`

---
## 1. Serveur MCP local

Le serveur MCP local est un processus Python indépendant lancé automatiquement par `MCPServerStdio`.
Il expose `rag_search`, l'équivalent MCP de `tool_retrieve_docs` codé en TP 3_1.

La différence clé avec TP 3_1 : les outils ne sont plus importés directement dans le notebook.
L'agent communique avec eux via le protocole MCP : texte structuré sur stdin/stdout.

Le serveur est défini dans `TP3_travel_planner_Agent/mcp_server.py`.

### 1.1. Compléter `rag_search` dans `mcp_server.py`

En vous inspirant de `tool_retrieve_docs` codé en TP 3_1.

---
## 2. Configuration des serveurs MCP

Quatre serveurs MCP sont utilisés :
- **local** : `mcp_server.py` exposant `rag_search` via `MCPServerStdio`
- **DeepWiki** : documentation de dépôts GitHub (sans clé API)
- **Tavily** : recherche et extraction web (clé `TAVILY_API_KEY`)
- **Google Maps via OpenMCP** : géocodage, lieux proches (clé `GOOGLE_API_KEY`)

Chaque serveur reçoit un `tool_prefix` : le nom de l'outil dans l'agent sera `{prefix}_{tool_name}`.

La construction exacte de `MCPServerStdio`/`MCPServerStreamableHTTP` (paramètres `command`/`args`/`cwd`/`url`/`headers`/`tool_prefix`) n'a pas de référence officielle stable et facile à localiser : le bloc de configuration est donc fourni tout fait ci-dessous, pas un exercice. La partie à comprendre/vérifier est la suite (section 3) : lire `server.list_tools()` et comparer les noms d'outils préfixés.

### 2.1. Config des serveurs MCP

In [ ]:
tavily_query = urlencode({"tavilyApiKey": project_settings.tavily_api_key})
tavily_url = f"https://mcp.tavily.com/mcp/?{tavily_query}"

google_maps_url = "https://mcp.open-mcp.org/api/server/google-maps@latest/mcp"
google_maps_headers = {"FORWARD_VAR_KEY": project_settings.google_api_geo_maps_key}

# Configuration fournie (pas de référence officielle stable pour ces deux classes, voir texte ci-dessus)

local_mcp_server = MCPServerStdio(
    command=sys.executable,
    args=["TP3_travel_planner_Agent/mcp_server.py"],
    cwd=str(ROOT_DIR),
    tool_prefix="local",
    timeout=20,
)

deepwiki_mcp_server = MCPServerStreamableHTTP(
    url="https://mcp.deepwiki.com/mcp",
    tool_prefix="deepwiki",
)

tavily_mcp_server = MCPServerStreamableHTTP(
    url=tavily_url,
    tool_prefix="tavily",
)

google_maps_mcp_server = MCPServerStreamableHTTP(
    url=google_maps_url,
    headers=google_maps_headers,
    tool_prefix="gmaps",
)

---
## 3. Connexion et création de l'agent

Configurer le modèle Google via PydanticAI, identique à TP 3_1.

### 3.1. Configurer le modèle

In [ ]:
model = GoogleModel(
    model_name=project_settings.llm_model_name,
    provider=GoogleProvider(api_key=project_settings.google_api_generative_key),
)

agent_model_settings = GoogleModelSettings(
    temperature=project_settings.llm_temperature,
    top_p=project_settings.llm_top_p,
    max_tokens=project_settings.llm_max_output_tokens,
    google_thinking_config={"thinking_budget": project_settings.llm_thinking_budget},
)

Le prompt système est adapté aux noms d'outils MCP (préfixés par `local_`, `deepwiki_`, `tavily_`, `gmaps_`).

Conseil : mentionner explicitement les noms des outils disponibles pour orienter l'agent.

Conseil : demandez à l'agent de **sourcer chaque fait** avec le nom de l'outil ou du document utilisé (ex: `[tool_name : source]`).

### 3.2. Rédiger le prompt système

In [ ]:
system_prompt = """

# RÈGLES

Tu es un assistant de planification de voyage orienté RAG.
Tu dois produire des recommandations personnalisées à partir de faits récupérés par outils.

## Règles générales

### Usage des outils
- ...

### Style de réponse
- ...

### Format attendu
1) ...

## Consignes spécifiques par outil

### local_rag_search
- ...

### tavily_search
- ...

### gmaps_geocode
- ...

"""

Avant de créer l'agent, vérifier que chaque serveur MCP répond correctement avec `server.list_tools()`.

Si un serveur est indisponible, une `ValueError` est levée avec le nom du serveur et le détail de l'erreur.

Résultat attendu :
```
[OK] local : 1 tools
[OK] deepwiki : 3 tools
[OK] tavily : 5 tools
[OK] gmaps : 18 tools
Serveurs MCP configurés : 4
```

### 3.3. Lister les serveurs et créer l'agent

In [ ]:
mcp_toolsets = [local_mcp_server, deepwiki_mcp_server, tavily_mcp_server, google_maps_mcp_server]

for server in mcp_toolsets:
    server_name = server.tool_prefix or server.__class__.__name__
    try:
        tools = await server.list_tools()
        print(f"[OK] {server_name} : {len(tools)} tools")
    except Exception as error:
        raise ValueError(
            f"Serveur MCP indisponible : {server_name} ({type(error).__name__} - {error})"
        ) from error

agent_mcp = LoggingAgent(
    model=model,
    instructions=system_prompt,
    model_settings=agent_model_settings,
    toolsets=mcp_toolsets,
)

print(f"Serveurs MCP configurés : {len(mcp_toolsets)}")

---
## 4. Cas d'usage 1 : Rome en 4 jours

### 4.1. Lancer l'agent

In [ ]:
request_use_case_1 = (
    "Je vais à Rome la semaine prochaine pour 4 jours (du jeudi au dimanche), "
    "arrivée le matin, départ le soir. "
    "Fais un plan de 4 jours avec un budget de 300 EUR pour les sorties et restaurants."
)

result_use_case_1 = await agent_mcp.run_with_logging(
    request=request_use_case_1,
    log_path=LOG_DIR / "trace_3_2_use_case_1.log",
    max_steps=12,
)

### 4.2. Afficher la réponse

In [ ]:
print(result_use_case_1.output)

--> Consultez aussi le fichier de log (`TP3_travel_planner_Agent/logs/trace_3_2_use_case_1.log`) : il contient la trace complète des outils appelés, dans quel ordre, avec quels arguments et résultats, ce qui permet de voir ce que l'agent en a déduit à chaque étape.

---
## 5. Cas d'usage 2 : Meilleure période Paris → New York

### 5.1. Lancer l'agent

In [ ]:
request_use_case_2 = (
    "Trouve la meilleure période dans les 6 prochains mois pour un voyage Paris → New York. "
    "Compare météo et prix saisonniers, et justifie la recommandation."
)

result_use_case_2 = await agent_mcp.run_with_logging(
    request=request_use_case_2,
    log_path=LOG_DIR / "trace_3_2_use_case_2.log",
    max_steps=12,
)

### 5.2. Afficher la réponse

In [ ]:
print(result_use_case_2.output)

---
## 6. Cas d'usage 3 : Recommandations près d'une adresse

### 6.1. Lancer l'agent

In [ ]:
request_use_case_3 = (
    "Recommande des restaurants et des activités près de cette adresse : "
    "10 Rue de la Paix, 75002 Paris, France. "
    "Je veux des options variées et un budget modéré."
)

result_use_case_3 = await agent_mcp.run_with_logging(
    request=request_use_case_3,
    log_path=LOG_DIR / "trace_3_2_use_case_3.log",
    max_steps=12,
)

### 6.2. Afficher la réponse

In [ ]:
print(result_use_case_3.output)